In [ ]:
import numpy as np
import scipy.ndimage as ndimage
from scipy.signal import spectrogram, resample_poly
from scipy.io import wavfile
import matplotlib.pyplot as plt

TARGET_SAMPLE_RATE = 16000 # CONFIGURE THIS <----------------------------------------------------------------------

# Audio to numpy samples
def audio_convert(file_path):
    # Reads the data from the file
    sample_rate, data = wavfile.read(file_path)

    # converts common stereo files into mono files by making sure its a 1D array
    if len(data.shape) > 1:
        data = np.mean(data, axis=1)

    # converts the files data into 16 bit integers and then to a decimal range between -1.0 and 1.0 to normalize the data
    normalized_samples = data.astype(np.float32) / 32768.0

    # scales the sampling rate to match the target
    if sample_rate != TARGET_SAMPLE_RATE:
        gcd = np.gcd(TARGET_SAMPLE_RATE, sample_rate)
        up = TARGET_SAMPLE_RATE // gcd
        down = sample_rate // gcd
        normalized_samples = resample_poly(normalized_samples, up, down)
        sample_rate = TARGET_SAMPLE_RATE

    return normalized_samples, sample_rate

# Audio snips to spectrogram conversion
def spectrogram_conversion(samples, sample_rate):
    # Creates a spectrogram with overlapping windowsizes so the audio doesn't have blips
    frequencies, times, spec = spectrogram(
        samples, fs=sample_rate, nperseg=2048, noverlap=1024
    )

    # Log scales the values, ensuring not to log 0
    log_spectro = np.log10(spec + 1e-10) * 10

    # Nearest neighbor grid
    neighborhood = np.ones((31, 31), dtype=bool)

    # Filter to gind the local high points in the landscape
    local_maxima_grid = ndimage.maximum_filter(log_spectro, footprint=neighborhood)
    
    print(np.min(log_spectro), np.max(log_spectro))
    # fixed absolute cutoff
    ABSOLUTE_DB_CUTOFF = -80

    # If it appears a lot (loudest point in its neighborhood) and is above the cut off its a peak
    peak_locations = (log_spectro == local_maxima_grid) & (log_spectro > ABSOLUTE_DB_CUTOFF)

    # Extracts the frequencies and time stamps
    frequency_indices, time_indicies = np.where(peak_locations)

    # Zips into a list of tuples
    extracted_peaks = list(zip(time_indicies, frequency_indices))

    # Chronological timeline
    extracted_peaks.sort()

    return log_spectro, extracted_peaks